# Traffic Demand — Train, Evaluate, Submit

Colab-friendly notebook mirroring `src/train.py` and `src/predict.py`.

In [ ]:
# Colab setup (uncomment if needed)
# !pip install -q -r ../requirements.txt

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config import MODEL_DIR, SUBMISSION_PATH
from src.data_loader import load_train, load_test, split_train_holdout
from src.features import FeatureEngineer
from src.train import train_models, fit_full_and_save
from src.predict import predict, validate_submission
from src.metrics import regression_metrics

## Load data & build features

In [ ]:
train_raw = load_train()
test_raw = load_test()
train_split, val_split = split_train_holdout(train_raw)

engineer = FeatureEngineer()
engineer.fit(train_split)
train_feat = engineer.transform(train_split)
val_feat = engineer.transform(val_split)
feature_cols = engineer.get_feature_columns()
print(f"Train: {len(train_feat)}, Val: {len(val_feat)}, Features: {len(feature_cols)}")

## Train CatBoost + LightGBM ensemble

In [ ]:
artifacts = train_models(train_feat, val_feat, feature_cols, use_log_target=True, tune=False)
print("Holdout metrics:", artifacts["metrics"])

## Optional: Optuna tuning (slower)

In [ ]:
# artifacts = train_models(train_feat, val_feat, feature_cols, use_log_target=True, tune=True)
# print("Tuned metrics:", artifacts["metrics"])

## Retrain on full data & save models

In [ ]:
engineer_full = FeatureEngineer()
engineer_full.fit(train_raw)
fit_full_and_save(train_raw, artifacts, engineer_full)
print(f"Saved to {MODEL_DIR}")

## Generate submission

In [ ]:
submission = predict(test_raw)
validate_submission(submission, test_raw)
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved {SUBMISSION_PATH} ({len(submission)} rows)")
submission.head()

## Optional: grid baseline / tiny CNN (GPU)

In [ ]:
from src.grid_model import train_grid_baseline, try_torch_cnn

grid_sub = train_grid_baseline(train_raw)
print("Grid day-48 baseline preview:", grid_sub["demand"].describe())

r2_cnn = try_torch_cnn(train_split, val_split)
print("Tiny CNN val R² (if torch installed):", r2_cnn)

## Feature importance (CatBoost)

In [ ]:
import json
from catboost import CatBoostRegressor

cat = CatBoostRegressor()
cat.load_model(str(MODEL_DIR / "catboost.cbm"))
meta = json.loads((MODEL_DIR / "train_meta.json").read_text())
fi = pd.DataFrame({
    "feature": meta["feature_cols"],
    "importance": cat.get_feature_importance(),
}).sort_values("importance", ascending=False)
fi.head(15)